# Natural language to SQL

**Run in [Google Colab](https://colab.research.google.com/) For GPU.**

This model have  Mistral as a base and it has been fine-tuned to excel in SQL code generation.

In [1]:
from google.colab import userdata
userdata.get('HF_TOKEN')

SecretNotFoundError: Secret HF_TOKEN does not exist.

In [ ]:
#Install the lastest versions of peft & transformers library recommended
#if you want to work with the most recent models
!pip install -q git+https://github.com/huggingface/peft.git
!pip install git+https://github.com/huggingface/accelerate.git
!pip install git+https://github.com/huggingface/transformers.git
!pip install bitsandbytes

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/huggingface/accelerate.git to /tmp/pip-req-build-j1562uuw
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/accelerate.git /tmp/pip-req-build-j1562uuw
  Resolved https://github.com/huggingface/accelerate.git to commit 917e2a9e37ba320c5800f11999f5c399508ed252
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for accelerate: filename=accelerate-1.14.0.dev0-py3-none-any.whl size=386923 sha256=ab6f8b0218db867cb13faf3692aa8910e5edebaba5e3df91a764c5d5b15e019b
  Stored in directory: /tmp/pip-ephem-wheel-cache-6sdq3m5u/wheels/5a/20/fb/1221fe933b56fe7ac69fd00159d9a1950bc8ced38198abc18f
Successfully built accelerate
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.13.0

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
import accelerate

In [ ]:
model_name = "defog/sqlcoder-7b"

We need to create the Quantization configuration to load the Model.

It is a large model and I want it to fit in a 16GB GPU, I'm going to use a 4 bits quantization.

If you want to learn more about quantization, refer to this article: [QLoRA: Training a Large Language Model on a 16GB GPU.](https://medium.com/towards-artificial-intelligence/qlora-training-a-large-language-model-on-a-16gb-gpu-00ea965667c1)

You can try to use this model in a 8 bit quantizations and check in you see any improvements in the results.

In [ ]:
bnb_config = BitsAndBytesConfig(
  load_in_4bit=True,
  bnb_4bit_use_double_quant=True,
  bnb_4bit_quant_type="nf4",
  bnb_4bit_compute_dtype=torch.bfloat16
)


To load the model I pass to the AutoModelForCasualLM teh quantization configurations, and HuggingFace take care of all the hard work.

In [ ]:
foundation_model = AutoModelForCausalLM.from_pretrained(model_name,
                    quantization_config=bnb_config,
                    device_map='auto',
                    use_cache = True)

config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

pytorch_model.bin.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
eos_token_id = tokenizer.convert_tokens_to_ids(["```"])[0]

tokenizer_config.json:   0%|          | 0.00/915 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

This function wraps the call to *model.generate*

In [ ]:
#this function returns the outputs from the model received, and inputs.
def get_outputs(model, inputs, max_new_tokens=400):
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        num_return_sequences=1,
        eos_token_id=eos_token_id,
        pad_token_id=eos_token_id,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=5
    )
    return outputs

# Prompt without Shots.
In this first PROMPT we are going to give Instructions to the model and pass the structure of the Database.

The instructions are significantly different from those we are passing to GPT-3.5-Turbo. This model is really well fine-tuned, but it is smaller than GPT-3.5.

We need to be more clear with the instructions, as it does not have the same capacity to understand our orders as GPT-3.5.

In [ ]:

    ### Instructions:
#Your task is convert a question into a SQL query, given a SQL database schema.
#Adhere to these rules:
#- **Deliberately go through the question and database schema word by word** to appropriately answer the question

    ### Input
 #   Generate a SQL query that answers the question below.
  #  This query will run on a database whose schema is represented in this string:
sp_nl2sql = """
    CREATE TABLE employees (
        ID_Usr INTEGER PRIMARY KEY,
        name VARCHAR(50),
        department VARCHAR(50),
        hire_date DATE
    );

    CREATE TABLE salary (
        ID_Usr INTEGER,
        year INTEGER,
        salary FLOAT,
        bonus FLOAT
    );

    CREATE TABLE studies (
        ID INTEGER PRIMARY KEY,
        ID_Usr INTEGER,
        educational_level INTEGER,
        Institution VARCHAR(100),
        Years INTEGER,
        Speciality VARCHAR(100)
    );

    ### Response
    Based on your instructions, here is the SQL query I have generated to answer the question
    `{question}`:
```sql3
"""

In [ ]:
sp_nl2sql = sp_nl2sql.format(question="Return the name of the best paid emplyee")
print(sp_nl2sql)


CREATE TABLE employees (
    ID_Usr INTEGER PRIMARY KEY,
    name VARCHAR(50),
    department VARCHAR(50),
    hire_date DATE
);

CREATE TABLE salary (
    ID_Usr INTEGER,
    year INTEGER,
    salary FLOAT,
    bonus FLOAT
);

CREATE TABLE studies (
    ID INTEGER PRIMARY KEY,
    ID_Usr INTEGER,
    educational_level INTEGER,
    Institution VARCHAR(100),
    Years INTEGER,
    Speciality VARCHAR(100)
);

### Response
Based on your instructions, here is the SQL query I have generated to answer the question
`Return the name of the best paid emplyee`:
```sql3



In [ ]:
input_sentences = tokenizer(sp_nl2sql, return_tensors="pt").to('cuda')
response = get_outputs(foundation_model, input_sentences, max_new_tokens=400)
SQL = tokenizer.batch_decode(response, skip_special_tokens=True)

In [ ]:
#Empty the cache in orde to do more calls without problems.
torch.cuda.empty_cache()

In [ ]:
print(SQL[0].split("```sql3")[-1].split("```")[0].split(";")[0].strip() + ";")

SELECT employees.name, MAX(salary.salary) AS max_salary FROM employees JOIN salary ON employees.id_usr = salary.id_usr GROUP BY employees.name ORDER BY max_salary DESC NULLS LAST LIMIT 1;


The SQL Order is correct.

#Prompt with shots OpenAI Style.
In this second prompt we are going to add some Shots with samples to see if our SQL style affects the model.

In [ ]:
sp_nl2sql2 = """
    ### Instructions:
Your task is convert a question into a SQL query, given a SQL database schema.
Adhere to these rules:
- **Deliberately go through the question and database schema word by word** to appropriately answer the question
- **Use the samples SQL in the ### Samples section to learn more about the Databases structure

    ### Input
    Generate a SQL query that answers the question below.
    This query will run on a database whose schema is represented in this string:

    CREATE TABLE employees (
        ID_Usr INTEGER PRIMARY KEY,
        name VARCHAR(50),
        department VARCHAR(50),
        hire_date DATE
    );

    CREATE TABLE salary (
        ID_Usr INTEGER,
        year INTEGER,
        salary FLOAT,
        bonus FLOAT
    );

    CREATE TABLE studies (
        ID INTEGER PRIMARY KEY,
        ID_Usr INTEGER,
        educational_level INTEGER,
        Institution VARCHAR(100),
        Years INTEGER,
        Speciality VARCHAR(100)
    );

    ### Response
    -- Q: What is the average salary per department?
    SELECT e.department, AVG(s.salary) AS avg_salary
    FROM employees e JOIN salary s ON e.ID_Usr = s.ID_Usr
    GROUP BY e.department;

    -- Q: Which institution has the highest average salary?
    SELECT st.Institution, AVG(sa.salary) AS avg_salary
    FROM studies st JOIN salary sa ON st.ID_Usr = sa.ID_Usr
    GROUP BY st.Institution ORDER BY avg_salary DESC LIMIT 1;

    `{question}`:
````sql3
    """

In [ ]:
sp_nl2sql2 = sp_nl2sql2.format(question="Return The name of the best paid employee")
(print(sp_nl2sql2))


    ### Instructions:
Your task is convert a question into a SQL query, given a SQL database schema.
Adhere to these rules:
- **Deliberately go through the question and database schema word by word** to appropriately answer the question
- **Use the samples SQL in the ### Samples section to learn more about the Databases structure

    ### Input
    Generate a SQL query that answers the question below.
    This query will run on a database whose schema is represented in this string:

    CREATE TABLE employees (
        ID_Usr INTEGER PRIMARY KEY,
        name VARCHAR(50),
        department VARCHAR(50),
        hire_date DATE
    );

    CREATE TABLE salary (
        ID_Usr INTEGER,
        year INTEGER,
        salary FLOAT,
        bonus FLOAT
    );

    CREATE TABLE studies (
        ID INTEGER PRIMARY KEY,
        ID_Usr INTEGER,
        educational_level INTEGER,
        Institution VARCHAR(100),
        Years INTEGER,
        Speciality VARCHAR(100)
    );

    ### Response
   

In [ ]:
input_sentences = tokenizer(sp_nl2sql2, return_tensors="pt").to('cuda')
response = get_outputs(foundation_model, input_sentences, max_new_tokens=400)
SQL = tokenizer.batch_decode(response, skip_special_tokens=True)
torch.cuda.empty_cache()

In [ ]:
print(SQL[0].split("```sql3")[-1].split("```")[0].split(";")[0].strip() + ";")

WITH ranked AS (SELECT e.name, e.department, s.institution, row_number() OVER (PARTITION BY e.department ORDER BY AVG(s.salary) DESC) AS rn
    FROM employees e JOIN salary s ON e.ID_Usr = s.ID_Usr
    GROUP BY e.name, e.department, s.institution)
    SELECT name, department, institution
    FROM ranked
    WHERE rn = 1;


The Order is really different from the one obtained with the first prompt.

The first difference is the format. But The SQL is realy more simple, at least it is my sensation.

#Prompt with Shots in Sample Style.

In this prompt, we will place the examples in a separate section, and in the instructions, we will instruct the model to pay attention to them in order to generate the SQL commands.

In [ ]:
sp_nl2sql3b = """
    ### Instructions:
Your task is convert a question into a SQL query, given a SQL database schema.
Adhere to these rules:
- **Deliberately go through the question and database schema word by word** to appropriately answer the question
- **Use the samples SQL in the ### Samples section to learn more about the Databases structure

    ### Input
    Generate a SQL query that answers the question below.
    This query will run on a database whose schema is represented in this string:

    CREATE TABLE employees (
        ID_Usr INTEGER PRIMARY KEY,
        name VARCHAR(50),
        department VARCHAR(50),
        hire_date DATE
    );

    CREATE TABLE salary (
        ID_Usr INTEGER,
        year INTEGER,
        salary FLOAT,
        bonus FLOAT
    );

    CREATE TABLE studies (
        ID INTEGER PRIMARY KEY,
        ID_Usr INTEGER,
        educational_level INTEGER,
        Institution VARCHAR(100),
        Years INTEGER,
        Speciality VARCHAR(100)
    );

    ### Samples

    -- Q: List all employees in the Engineering department.
    SELECT name FROM employees WHERE department = 'Engineering';

    -- Q: What is the total bonus paid in 2023?
    SELECT SUM(bonus) FROM salary WHERE year = 2023;

    -- Q: Which employees have a PhD?
    SELECT e.name FROM employees e
    JOIN studies st ON e.ID_Usr = st.ID_Usr
    WHERE st.educational_level = 3;

    ### Response
    Based on your instructions, here is the SQL query I have generated to answer the question
    `{question}`:
```sql3
    """

In [ ]:
sp_nl2sql3 = sp_nl2sql3b.format(question="Return The name of the best paid employee")
print (sp_nl2sql3)


    ### Instructions:
Your task is convert a question into a SQL query, given a SQL database schema.
Adhere to these rules:
- **Deliberately go through the question and database schema word by word** to appropriately answer the question
- **Use the samples SQL in the ### Samples section to learn more about the Databases structure

    ### Input
    Generate a SQL query that answers the question below.
    This query will run on a database whose schema is represented in this string:

    CREATE TABLE employees (
        ID_Usr INTEGER PRIMARY KEY,
        name VARCHAR(50),
        department VARCHAR(50),
        hire_date DATE
    );

    CREATE TABLE salary (
        ID_Usr INTEGER,
        year INTEGER,
        salary FLOAT,
        bonus FLOAT
    );

    CREATE TABLE studies (
        ID INTEGER PRIMARY KEY,
        ID_Usr INTEGER,
        educational_level INTEGER,
        Institution VARCHAR(100),
        Years INTEGER,
        Speciality VARCHAR(100)
    );

    ### Samples

   

In [ ]:
input_sentences = tokenizer(sp_nl2sql3, return_tensors="pt").to('cuda')
response = get_outputs(foundation_model, input_sentences, max_new_tokens=400)
SQL = tokenizer.batch_decode(response, skip_special_tokens=True)
torch.cuda.empty_cache()

In [ ]:
print(SQL[0].split("```sql3")[-1].split("```")[0].split(";")[0].strip() + ";")

SELECT MAX(salary.salary) AS max_salary, employees.name FROM salary JOIN employees ON salary.ID_Usr = employees.ID_Usr GROUP BY employees.name ORDER BY max_salary DESC NULLS LAST LIMIT 1;


#Now the question in spanish.


In [ ]:
sp_nl2sql3 = sp_nl2sql3b.format(question="Qual o funcionario mais bem pago?")
print (sp_nl2sql3)


    ### Instructions:
Your task is convert a question into a SQL query, given a SQL database schema.
Adhere to these rules:
- **Deliberately go through the question and database schema word by word** to appropriately answer the question
- **Use the samples SQL in the ### Samples section to learn more about the Databases structure

    ### Input
    Generate a SQL query that answers the question below.
    This query will run on a database whose schema is represented in this string:

    CREATE TABLE employees (
        ID_Usr INTEGER PRIMARY KEY,
        name VARCHAR(50),
        department VARCHAR(50),
        hire_date DATE
    );

    CREATE TABLE salary (
        ID_Usr INTEGER,
        year INTEGER,
        salary FLOAT,
        bonus FLOAT
    );

    CREATE TABLE studies (
        ID INTEGER PRIMARY KEY,
        ID_Usr INTEGER,
        educational_level INTEGER,
        Institution VARCHAR(100),
        Years INTEGER,
        Speciality VARCHAR(100)
    );

    ### Samples

   

In [ ]:
input_sentences = tokenizer(sp_nl2sql3, return_tensors="pt").to('cuda')
response = get_outputs(foundation_model, input_sentences, max_new_tokens=400)
SQL = tokenizer.batch_decode(response, skip_special_tokens=True)
torch.cuda.empty_cache()

In [ ]:
print(SQL[0].split("```sql3")[-1].split("```")[0].split(";")[0].strip() + ";")

WITH total_salary AS (SELECT id_usr, SUM(salary) AS total_salary FROM salary GROUP BY id_usr), total_bonus AS (SELECT id_usr, SUM(bonus) AS total_bonus FROM salary GROUP BY id_usr) SELECT e.name, ts.total_salary, tb.total_bonus FROM employees e JOIN total_salary ts ON e.id_usr = ts.id_usr JOIN total_bonus tb ON e.id_usr = tb.id_usr;


The generated SQL command is the same regardless of where we have placed the examples.

#Conclusions.

Let's see the three SQL's together.

* SELECT employees.name, MAX(salary.salary) AS max_salary FROM employees JOIN salary ON employees.ID_Usr = salary.ID_Usr GROUP BY employees.name ORDER BY max_salary DESC NULLS LAST LIMIT 1;

* SELECT e.name
    FROM employees e
    JOIN salary s ON e.ID_Usr = s.ID_usr
    WHERE s.salary = (SELECT MAX(salary) FROM salary);

* SELECT e.name
    FROM employees e
    JOIN salary s ON e.ID_Usr = s.ID_usr
    WHERE s.salary = (SELECT MAX(salary) FROM salary);

* Spanish Question: SELECT e.name
     FROM employees e
     JOIN salary s ON e.ID_Usr = s.ID_Usr
     WHERE s.salary = (SELECT MAX(salary) FROM salary)
     GROUP BY e.name
     ORDER BY COUNT(studies.ID_study) DESC
     LIMIT 1;


**The model has demonstrated that it is highly efficient in crafting SQL.** Additionally, it pays a lot of attention, perhaps too much, to the examples we provide. Clearly, these examples should be crafted by one of the best SQL programmers we have access to, though their use may not be essential.

On the other hand, although the model is clearly very proficient in SQL generation, during the creation of the notebook, I have encountered several issues because the commands need to be extremely clear. It doesn't handle typos well (which should not exist).

It appears to have some issues when it receives commands in Spanish. I assume this problem would be present in any language other than English. Therefore, since it's a tool that could be used by non-technical personnel, this should be considered in environments where English is not the primary language.

# Exercise
 - Complete the prompts similar to what we did in class.
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [ ]:
#  pergunta simples, 1 tabela
v1 = sp_nl2sql3b.format(question="How many employees were hired after 2019?")
input_sentences = tokenizer(v1, return_tensors="pt").to('cuda')
response = get_outputs(foundation_model, input_sentences, max_new_tokens=400)
SQL = tokenizer.batch_decode(response, skip_special_tokens=True)
torch.cuda.empty_cache()

print("V1:", SQL[0].split("```sql3")[-1].split("```")[0].split(";")[0].strip() + ";")

In [ ]:
# multi-tabela com filtro
v2 = sp_nl2sql3b.format(question="List the names and salaries of employees with a Master's degree in 2023, ordered by salary descending")
input_sentences = tokenizer(v2, return_tensors="pt").to('cuda')
response = get_outputs(foundation_model, input_sentences, max_new_tokens=400)
SQL = tokenizer.batch_decode(response, skip_special_tokens=True)
torch.cuda.empty_cache()

print("V2:", SQL[0].split("```sql3")[-1].split("```")[0].split(";")[0].strip() + ";")

In [ ]:
# pergunta fora do schema (testa alucinação)
v3 = sp_nl2sql3b.format(question="What is the performance review score for each employee?")
input_sentences = tokenizer(v3, return_tensors="pt").to('cuda')
response = get_outputs(foundation_model, input_sentences, max_new_tokens=400)
SQL = tokenizer.batch_decode(response, skip_special_tokens=True)
torch.cuda.empty_cache()

print("V3:", SQL[0].split("```sql3")[-1].split("```")[0].split(";")[0].strip() + ";")

The zero-shot prompt already produced valid SQL for simple queries, but adding few-shot samples
in a dedicated ### Samples section improved accuracy significantly for multi-table JOINs.
The model struggled with non-English questions, sometimes mixing column names or adding incorrect
clauses, confirming SQLCoder is optimized for English. The key lesson is that for fine-tuned SQL
models, prompt structure and language matter as much as the schema definition itself.